In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import correlate2d , convolve2d

In [ ]:
alpha = 0.01
learning_rate = 0.01

In [ ]:
def leaky_relu(z: np.array):
    return np.where(z > 0 , z , alpha * z)

In [ ]:
class Convolution:
    input_size = (0,0,0)
    filter_ze = 0
    output_shape = (0,0, 0)
    filter_shape = (0,0,0)
    no_of_filters = 0
    input_data = None 
    def __int__(self, input_size , filter_size  , no_of_filters):
        input_height, input_width = input_size
        self.input_size = input_size
        self.filter_size = filter_size 
        self.no_of_filters = no_of_filters
        
        self.filter_shape = ( no_of_filters, input_height , input_width )
        self.output_size = (no_of_filters,input_height-self.filter_size+1 , input_width - self.filter_size+1 ) 
        self.filters = np.random.randn(*self.filter_shape)
        self.bias = np.random.random(*self.output_shape)
    
    def forward_propagation(self ,data : np.matrix ):
        output = np.zeros(self.output_shape)
        self.input_data = data
        for i in self.no_of_filters:
            output[i] = correlate2d(self.input_data, self.filters[i] , mode="valid")
        output = leaky_relu(output)
        return output  
    
    def back_propagation( self , dl_do : np.matrix) -> np.matrix :
        dl_dfilters = np.zeros_like(self.filters)
        dl_dinput = np.zeros_like(self.input_data)
        for i in self.no_of_filters:
           dl_dfilters[i] = correlate2d(self.input_data , dl_do[i] , mode="valid")
           dl_dinput[i] = convolve2d(dl_do[i],self.filters[i] , mode= "full")
        self.filters -= learning_rate * dl_dfilters
        self.bias -= learning_rate * dl_do
        return dl_dinput

        

In [ ]:
class MaxPooling :
    pool_size = 0
    no_of_filters = 0 
    output_size = ()
    def __int__(self , pool_size  ):
        self.pool_size = (pool_size , pool_size) 
    def forward_propgataion(self, featureMap: np):
        self.no_of_filters, self.height , self.width = featureMap.shape
        self.output_size = (self.no_of_filters, self.height // self.pool_size , self.width // self.pool_size)
        self.output = np.zeros(self.output_size)
        self.input_data = featureMap
        for i in self.no_of_filters: 
            for height in range(self.output_size[1]):
                for width in range(self.output_size[2]):
                    height_start = height * self.pool_size
                    width_start =  width * self.pool_size  

                    height_end = height_start + self.pool_size
                    width_end = width_start + self.pool_size 

                    self.output[i][height][width] = np.max(input[i , height_start : height_end , width_start : width_end])
        return self.output
        
    def back_propagation(self , dl_do: np.matrix) -> np.matrix:
        dl_dx = np.zeros_like(self.input_data) 
        
        for i in self.no_of_filters:
            for height in self.output_size[1] :
                 for width in self.output_size[2]:
                    height_start = height * self.pool_size
                    width_start =  width * self.pool_size  

                    height_end = height_start + self.pool_size
                    width_end = width_start + self.pool_size 
                    patch = self.input_data[i , height_start : height_end , width_start: width_end]
                    mask = patch == np.max(patch)  

                    dl_dx[i , height_start: height_end , width_start : width_end ] = dl_do[i][height][width] * mask
        return mask
    
